In [ ]:
# https://medium.com/@ayush4002gupta/building-an-llm-agent-to-directly-interact-with-a-database-0c0dd96b8196

import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage

load_dotenv()  # This looks for the .env file and loads it into os.environ

llm = ChatOpenAI(
    model="gpt-4o-mini",  # recommended for tools + cost
    api_key=os.environ["API_KEY"],
    temperature=0,
    seed=100,
)

response = llm.invoke([
    HumanMessage(content="Reply with exactly: OK")
])

print(response.content)

DB_LIST_ANDROID = [
  r"D:\ExtractedDBs\core.db",
  r"D:\ExtractedDBs\journal.db",
  r"D:\ExtractedDBs\main.db",
  r"D:\ExtractedDBs\memories.db",
  r"D:\ExtractedDBs\commerce.db",
  r"D:\ExtractedDBs\msgstore.db",
  r"D:\ExtractedDBs\wa.db",
  r"D:\ExtractedDBs\account1cache4.db",
  r"D:\ExtractedDBs\account2cache4.db",
  r"D:\ExtractedDBs\account3cache4.db",
  r"D:\ExtractedDBs\gmm_storage.db",
  r"D:\ExtractedDBs\gmm_myplaces.db",
  r"D:\ExtractedDBs\peopleCache_sharononeil368@gmail.com_com.google_14.db",
  r"D:\ExtractedDBs\SBrowser.db",
  r"D:\ExtractedDBs\SBrowser2.db",
  r"D:\ExtractedDBs\searchengine.db"
]
DB_LIST_IPHONE = [
  r"D:\ExtractedDBs\ChatStorage.sqlite",
  r"D:\ExtractedDBs\ContactsV2.sqlite",
  r"D:\ExtractedDBs\CallHistory.sqlite",
  r"D:\ExtractedDBs\AddressBook.sqlitedb",
  r"D:\ExtractedDBs\AddressBookImages.sqlitedb",
  r"D:\ExtractedDBs\sms.db",
  r"D:\ExtractedDBs\Bookmarks.db",
  r"D:\ExtractedDBs\CloudTabs.db",
  r"D:\ExtractedDBs\History.db",
  r"D:\ExtractedDBs\Calendar.sqlitedb",
  r"D:\ExtractedDBs\Extras.db"
]

ENTITY_CONFIG = {
    "email addresses": {
        # "regex": r"[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}",
        "regex": r"(?i)\b[a-z0-9][a-z0-9._%+-]{0,63}@a-z0-9?(?:.a-z0-9?)+\b",
        "desc": "valid electronic mail formats used for account registration or contact"
    },
    "phone numbers": {
        # "regex": r"\+?[0-9]{1,4}[- .]?\(?[0-9]{1,3}?\)?[- .]?[0-9]{1,4}[- .]?[0-9]{1,4}[- .]?[0-9]{1,9}",
        "regex": r"(?x)\b(?:+|00)?[1-9]\d{0,2}[\s.-]?(?:(\d{1,4})|\d{1,4})[\s.-]?\d{2,4}(?:[\s.-]?\d{2,4}){1,3}(?:\s*(?:ext.?|x)\s*\d{1,6})?\b",
        "desc": "international or local telephone numbers"
    },
    "usernames": {
        # "regex": r"\b[a-zA-Z][a-zA-Z0-9._-]{2,31}\b",
        "regex": r"\b[a-zA-Z0-9][a-zA-Z0-9._-]{1,30}[a-zA-Z0-9]\b",
        "desc": "application-specific usernames, handles, or account identifiers"
    },
    "real names": {
        # "regex": r"[A-Za-z\u4e00-\u9fff][A-Za-z\u4e00-\u9fff\s\.\-]{1,50}",
        "regex": r"(?:\b(?:[A-Z][a-z]+(?:[-'][A-Z][a-z]+)?)(?:\s+(?:[A-Z][a-z]+|[A-Z].)){1,3}\b|[\u4e00-\u9fff]{2,4})",
        "desc": (
            "loosely structured human name-like strings used only for discovery "
            "and column pre-filtering; final identification is performed during extraction"
        )
    }
}

In [ ]:
# Core Python
import sqlite3
import re
import json
import time
from typing import TypedDict, Optional, List, Annotated
from langgraph.graph.message import add_messages

# LangChain / LangGraph
from langchain_core.tools import tool
from langchain_core.messages import (
    HumanMessage,
    AIMessage,
    SystemMessage
)
from langchain.agents import create_agent
from langgraph.graph import StateGraph, END
from langgraph.graph.message import MessagesState

def normalize_sql(sql: str) -> str:
    if not sql:
        return sql

    sql = sql.strip()

    # Remove ```sql or ``` fences
    sql = re.sub(r"^```(?:sql)?", "", sql, flags=re.IGNORECASE).strip()
    sql = re.sub(r"```$", "", sql).strip()

    # Remove leading 'sql' token if present
    if sql.lower().startswith("sql"):
        sql = sql[3:].strip()

    return sql

def safe_json_loads(text: str, default):
    if not text:
        return default

    text = text.strip()

    # Remove markdown fences
    if text.startswith("```"):
        parts = text.split("```")
        if len(parts) >= 2:
            text = parts[1].strip()

    # Remove leading 'json' token
    if text.lower().startswith("json"):
        text = text[4:].strip()

    try:
        return json.loads(text)
    except Exception as e:
        print("[JSON PARSE ERROR]")
        print("RAW:", repr(text))
        print("ERROR:", e)
        return default

def regexp(expr, item):
    # 1. Handle NULLs (None in Python)
    if item is None:
        return False

    try:
        # 2. Ensure item is a string (handles BLOBs/Bytes)
        if isinstance(item, bytes):
            item = item.decode('utf-8', errors='ignore')
        else:
            item = str(item)

        # 3. Compile and search
        return re.search(expr, item, re.IGNORECASE) is not None
    except Exception:
        # 4. If anything else goes wrong, don't crash SQLite
        return False


In [ ]:
@tool
def list_tables() -> str:
    """
    List all table names in the SQLite database.
    """
    conn = sqlite3.connect(DB_PATH)
    cur = conn.cursor()
    cur.execute("SELECT name FROM sqlite_master WHERE type='table'")
    tables = [r[0] for r in cur.fetchall()]
    conn.close()
    return ", ".join(tables)


@tool
def get_schema(table: str) -> str:
    """
    Return column names and types for a table.
    """
    conn = sqlite3.connect(DB_PATH)
    cur = conn.cursor()
    cur.execute(f"PRAGMA table_info('{table}')")
    cols = cur.fetchall()
    conn.close()
    return ", ".join(f"{c[1]} {c[2]}" for c in cols)


@tool
def exec_sql(query: str) -> dict:
    """Execute SQL statements. If one fails, it is skipped and the next is executed."""
    query_text = normalize_sql(query)
    queries = [q.strip() for q in query_text.split(";") if q.strip()]

    all_rows = []
    column_names = []
    conn = sqlite3.connect(DB_PATH)
    conn.create_function("REGEXP", 2, regexp)
    cur = conn.cursor()

    for q in queries:
        try:
            print(f"[EXECUTE] Running: {q}")
            cur.execute(q)

            # Update columns only if this specific query returns data
            if cur.description:
                column_names = [d[0] for d in cur.description]

            all_rows.extend(cur.fetchall())
        except Exception as e:
            # We log the error to the console for your situational awareness
            print(f"[SQL ERROR - SKIPPING]: {e}")
            # 'continue' ensures we skip the rest of this loop iteration
            # and move to the next query in the list
            continue

    conn.close()
    return {
        "rows": all_rows,
        "columns": column_names
    }

In [ ]:
class EmailEvidenceState(TypedDict):
    messages: Annotated[list, add_messages]
    attempt: int
    max_attempts: int
    phase: str            # "discovery" | "extraction"
    sql: Optional[str]
    rows: Optional[List]
    classification: Optional[dict]
    evidence: Optional[List[str]]
    target_entity: str     # <--- ADD THIS LINE
    # Add this to track the "winning" columns
    source_columns: Optional[List[dict]]
    # SQL used during discovery that returned results
    discovered_sql: Optional[List[str]]


def get_discovery_system(target, regex):
    return SystemMessage(
        content=(
            "You are a SQL planner. You are provided databases that are extracted from Android or iOS devices.\n"
            f"Goal: discover if any column contains {target}.\n\n"
            "Rules:\n"
            "- Use 'REGEXP' OR 'LIKE' for pattern matching.\n"
            f"- Example: SELECT col FROM table WHERE col REGEXP '{regex}' LIMIT 10\n"
            "- Validate your SQL and make sure all tables and columns do exist.\n"
            "- If multiple SQL statements are provided, combine them using UNION ALL.\n"
            "- Return ONLY SQL."
        )
    )

def get_extraction_system(target):
    return SystemMessage(
        content=(
            "You are a SQL refiner.\n"
            f"Goal: modify previously successful SQL to extract ALL {target}.\n\n"
            "Rules:\n"
            "- Validate each SQL and make sure all tables and columns do exist. Do NOT invent new tables or columns. ONLY use tables and columns that were confirmed in the conversation history.\n"
            "- Remove LIMIT clauses.\n"
            "- Preserve WHERE conditions and REGEXP filters.\n"
            "- Generate a single SQL statement to pull ALL rows from the confirmed source (no explanations)."
            "- If you need data from multiple tables, use UNION ALL.\n"
            "- Return ONLY SQL."
        )
    )

In [ ]:
def planner(state: EmailEvidenceState):
    # 1. Pre-fetch the "Grounding" info
    tables = list_tables.invoke({})
    config = ENTITY_CONFIG.get(state["target_entity"], ENTITY_CONFIG["email addresses"])

    # 2. Build the intelligent prompt
    if state["phase"] == "discovery":
        base_system = get_discovery_system(state["target_entity"], config["regex"])
    else:
        base_system = get_extraction_system(state["target_entity"])

    # Inject the actual database reality into the system message
    grounded_content = (
        f"{base_system.content}\n\n"
        f"--- DATABASE REALITY ---\n"
        f"EXISTING TABLES: {tables}\n"
        f"CURRENT PHASE: {state['phase']}\n"
        f"MAX ATTEMPTS: {state['max_attempts']}\n"
        "CRITICAL: Do not query tables that do not exist in the list above."
    )

    # 3. Use the Agent to reason with this information
    agent = create_agent(llm, [list_tables, get_schema])

    # Prepend the grounded system message to the history
    result = agent.invoke({"messages": [SystemMessage(content=grounded_content)] + state["messages"]})

    sql = normalize_sql(result["messages"][-1].content)

    attempt = state["attempt"]

    if state["phase"] == "discovery":
        attempt += 1
        print(f"[PLANNER] DISCOVERY MODE: attempt {attempt} of {state['max_attempts']}")
    else:
        print(f"[PLANNER] EXTRACTION MODE: Processing confirmed source...")

    return {
        "messages": [AIMessage(content=sql)],
        "sql": sql,
        "attempt": attempt
    }


def sql_execute(state: EmailEvidenceState):
    # Call the tool (it now returns a dict)
    result = exec_sql.invoke(state["sql"])

    rows = result.get("rows", [])
    cols = result.get("columns", [])

    print(f"[SQL EXEC] Retrieved {(rows)}")
    updates = {
        "rows": rows,
        "messages": [AIMessage(content=f"Retrieved {len(rows)} rows")]
    }

    # Tracking logic: Save columns to state only during extraction
    if state["phase"] == "extraction":
        updates["source_columns"] = cols
        print(f"[TRACKING] Saved source columns: {cols}")

    return updates


def rows_to_text(rows, limit=None, max_chars=5000, cell_max=1000):
    """
    Converts SQL rows to text with safety limits for LLM context.
    - limit: Max number of rows to process.
    - max_chars: Hard limit for the total string length.
    - cell_max: Max length for any single column value.
    """
    if not rows:
        return ""

    out = []
    # 1. Row-level limiting
    target_rows = rows[:limit] if limit else rows

    for r in target_rows:
        row_str = ",".join(
            (str(c)[:cell_max] + "..." if len(str(c)) > cell_max else str(c))
            for c in r if c is not None
        )
        out.append(row_str)

    final_text = "\n".join(out)

    # 2. Final global character limit safety check
    if len(final_text) > max_chars:
        return final_text[:max_chars] + "\n... [DATA TRUNCATED] ..."

    # print(f"[ROWS TO TEXT] Input: {len(rows)} rows | Output: {len(final_text)} chars")
    # Optional: print only the first 200 characters of the text to keep logs clean
    # print(f"[PREVIEW]: {final_text[:200]}...")
    return final_text


def get_classify_system(target: str):
    return SystemMessage(
        content=(
            f"Decide whether the text contains {target}.\n"
            "Return ONLY a JSON object with these keys:\n"
            "{ \"found\": true/false, \"confidence\": number, \"reason\": \"string\" }"
        )
    )


def classify(state: EmailEvidenceState):
    # 1. Prepare the text sample for the LLM
    text = rows_to_text(state["rows"], limit=10)

    # 2. Get the target-specific system message
    system_message = get_classify_system(state["target_entity"])

    # 3. Invoke the LLM
    result = llm.invoke([
        system_message,
        HumanMessage(content=f"Data to analyze:\n{text}")
    ]).content

# 4. Parse the decision
    decision = safe_json_loads(
        result,
        default={"found": False, "confidence": 0.0, "reason": "parse failure"}
    )

    # print("[CLASSIFY]", decision)
    return {"classification": decision}


def switch_to_extraction(state: EmailEvidenceState):
    print("[PHASE] discovery → extraction")
    return {"phase": "extraction"}


def extract(state: EmailEvidenceState):
    text = rows_to_text(state["rows"])
    system = f"Extract and normalize {state['target_entity']} from text. Return ONLY a JSON array of strings."
    result = llm.invoke([SystemMessage(content=system), HumanMessage(content=text)]).content
    return {"evidence": safe_json_loads(result, default=[])}


def next_step(state: EmailEvidenceState):
    # Once in extraction phase, extract and stop
    if state["phase"] == "extraction":
        return "do_extract"

    c = state["classification"]

    if c["found"] and c["confidence"] >= 0.6:
        return "to_extraction"

    if not c["found"] and c["confidence"] >= 0.6:
        return "stop_none"

    if state["attempt"] >= state["max_attempts"]:
        return "stop_limit"

    return "replan"

In [ ]:
def observe(state: EmailEvidenceState):
    """
    Debug / inspection node.
    Does NOT modify state.
    """

    print("\n=== STATE SNAPSHOT ===")

    # Messages
    print("\n--- MESSAGES ---")
    for i, m in enumerate(state["messages"]):
        print(f"{i}: {m.type.upper()} -> {m.content}")

    # Metadata
    print("\n--- BEGIN METADATA ---")
    print(f"attempt       : {state['attempt']}")
    print(f"max_attempts : {state['max_attempts']}")
    print(f"phase         : {state['phase']}")
    print(f"sql           : {state['sql']}")
    print(f"rows          : {state['rows']}")
    print(f"classification: {state['classification']}")
    print(f"evidence        : {state['evidence']}")

    print(f"Source Columns: {state.get('source_columns')}")
    print("\n--- END METADATA ---")

    # IMPORTANT: do not return state, return no-op update
    return {}



from langgraph.graph import StateGraph, END

graph = StateGraph(EmailEvidenceState)

# Define nodes (reusing the same 'observe' function for two different node names)
graph.add_node("planner", planner)
graph.add_node("observe_plan", observe)      # Checkpoint 1: The SQL Plan
graph.add_node("execute", sql_execute)
graph.add_node("classify", classify)
graph.add_node("observe_classify", observe)  # Checkpoint 2: The Logic/Discovery
graph.add_node("switch_phase", switch_to_extraction)
graph.add_node("extract", extract)
graph.add_node("observe_final", observe)     # Checkpoint 3: The Results

graph.set_entry_point("planner")

# --- THE FLOW ---
graph.add_edge("planner", "observe_plan")   # Check SQL before running
graph.add_edge("observe_plan", "execute")

graph.add_edge("execute", "classify")
graph.add_edge("classify", "observe_classify")

# The decision logic now triggers after the second observation
graph.add_conditional_edges(
    "observe_classify",  # Must match the new node name exactly
    next_step,
    {
        "to_extraction": "switch_phase",
        "do_extract": "extract",
        "replan": "planner",
        "stop_none": END,
        "stop_limit": END,
    }
)

graph.add_edge("switch_phase", "planner")

# Change this: Route 'extract' to our new observer instead of END
graph.add_edge("extract", "observe_final")
graph.add_edge("observe_final", END)

app = graph.compile()





In [ ]:
# Set your target here once
TARGET = "email addresses"
# TARGET = "phone numbers"
# TARGET = "usernames"
# TARGET = "real names"

print(f"Starting Batch Process for target: {TARGET}\n")

for i, db_file in enumerate(DB_LIST_ANDROID):
    print("="*50)
    print(f"Processing Database ({i+1}/{len(DB_LIST_ANDROID)}): {db_file}")

    # [CRITICAL] Update the global variable so the tools connect to the new DB
    # This works because the tools look up 'DB_PATH' at runtime.
    DB_PATH = db_file

    # Check if file exists to prevent crashing the loop
    if not os.path.exists(DB_PATH):
        print(f"File not found: {DB_PATH}. Skipping...")
        continue

    try:
        # 3. Invoke the Agent
        # We pass a fresh state for each database
        result = app.invoke({
            "messages": [HumanMessage(content=f"Find {TARGET} in the database")],
            "attempt": 0,
            "max_attempts": 3,
            "phase": "discovery",
            "target_entity": TARGET,
            "sql": None,
            "rows": None,
            "classification": None,
            "evidence": [],
            "source_columns": []
        })

        # 4. Prepare JSON Output
        final_evidence = result.get("evidence", [])
        output_data = {
            "database_file": db_file,
            "target_entity": TARGET,
            "status": "success" if final_evidence else "no_evidence_found",
            "evidence_count": len(final_evidence),
            "evidence": final_evidence,
            "source_columns": result.get("source_columns", []),
            "attempts_used": result.get("attempt"),
            "final_phase": result.get("phase")
        }

        # 5. Construct Filename and Save
        # Format: DBName_Entity.json (Sanitized)
        safe_db = os.path.basename(db_file).replace(" ", "_").replace(".sqlite", "").replace(".db", "")
        safe_target = TARGET.replace(" ", "_")
        filename = fr"C:\Users\SAfolabi\Code\results-android\usernames\{safe_db}_{safe_target}.json"

        with open(filename, "w", encoding="utf-8") as f:
            json.dump(output_data, f, indent=4)

        print(f"Results saved to: {filename}")
        print(f"Found {len(final_evidence)} items.")

    except Exception as e:
        print(f"Error processing {db_file}: {e}")
        # Optional: Save an error log JSON so you know it failed
        error_data = {"database": db_file, "error": str(e)}
        with open(f"ERROR_{db_file}.json", "w") as f:
            json.dump(error_data, f)

    # 6. Time Delay (1 minute)
    # Skip delay after the very last item
    if i < len(DB_LIST_ANDROID) - 1:
        print("Waiting 30 seconds before next database...")
        time.sleep(30)

print("\n" + "="*50)
print("Processing Complete.")
